# 🤖 Agentic AI Pipeline

A step up from a single rule-based agent: this notebook implements a small but
complete **agentic pipeline** with the classic loop:

```
User query -> Planner -> Tool Executor -> Memory -> Response Formatter
```

- **Planner** breaks a (possibly compound) query into an ordered list of steps,
  each step assigned to a tool.
- **Tool Executor** runs each step, one at a time, with per-step error handling
  (one failing step doesn't kill the whole run).
- **Memory** keeps a running log of every turn (query, plan, results) so the
  agent has conversational context and a debuggable trace.
- **Response Formatter** returns one structured JSON object per turn.

It supports compound requests like *"Calculate 12 * 4 and extract keywords
from this sentence about markets"* by planning **two** steps and executing
both, instead of only ever matching a single intent.

## 🛠️ Step 1: Tools

Four independent tools the agent can call.

In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression safely (digits + + - * / % ( ) . only)."""
    import re
    expr = expression.strip()
    if not expr:
        return "Error: empty expression"
    if not re.fullmatch(r"[0-9\.\s\+\-\*\/\(\)%]+", expr):
        return f"Error: unsupported characters in '{expression}'"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error in calculation: {e}"


In [2]:
# 🛠️ TOOL 2: Keyword extractor

def extract_keywords(text: str, top_n: int = 5) -> list:
    """Extract up to top_n unique keywords (words longer than 4 chars) from text."""
    try:
        words = text.split()
        keywords = list(dict.fromkeys(
            w.strip(".,!?;:\"'").lower() for w in words if len(w) > 4
        ))
        return keywords[:top_n]
    except Exception:
        return []


In [3]:
# 🛠️ TOOL 3: Summarizer (simple extractive summary, no external APIs)

def summarize(text: str, max_sentences: int = 2) -> str:
    """Return the first max_sentences sentences of text as a lightweight summary."""
    import re
    try:
        sentences = re.split(r"(?<=[.!?])\s+", text.strip())
        sentences = [s for s in sentences if s]
        if not sentences:
            return "Nothing to summarize."
        return " ".join(sentences[:max_sentences])
    except Exception as e:
        return f"Error summarizing: {e}"


In [4]:
# 🛠️ TOOL 4: Knowledge lookup (small local knowledge base, fallback tool)

KNOWLEDGE_BASE = {
    "python": "Python is a high-level, general-purpose programming language known for readability.",
    "agent": "An AI agent perceives input, plans actions, uses tools, and acts toward a goal.",
    "llm": "A large language model (LLM) predicts text based on patterns learned from data.",
    "api": "An API (application programming interface) lets software components talk to each other.",
}

def knowledge_lookup(term: str) -> str:
    """Look up a short definition for a known term; otherwise say it's unknown."""
    key = term.strip().lower()
    return KNOWLEDGE_BASE.get(key, f"No local definition found for '{term}'.")


## 🧭 Step 2: Memory

A tiny in-notebook memory store. In a production system this would be a database or vector store; here it's a list that persists for the life of the kernel session.

In [5]:
# 🧭 MEMORY

from datetime import datetime

class Memory:
    """Stores the full trace of every turn: query, plan, step results, final response."""

    def __init__(self):
        self.history = []

    def log(self, query: str, plan: list, steps: list, response: dict):
        self.history.append({
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "query": query,
            "plan": plan,
            "steps": steps,
            "response": response,
        })

    def recent(self, n: int = 5) -> list:
        return self.history[-n:]

    def clear(self):
        self.history = []


## 🧩 Step 3: Planner

The planner splits a query on connector words (`and`, `then`, `,`) into segments, and assigns each segment to a tool based on intent keywords. This is what makes it *agentic* rather than a single if/elif: one query can produce a multi-step plan.

In [6]:
# 🧩 PLANNER

import re

INTENT_KEYWORDS = {
    "calculator": ["calculate", "compute"],
    "keywords": ["keyword", "keywords"],
    "summarize": ["summarize", "summarise", "tl;dr", "shorten"],
    "knowledge": ["define", "what is a", "what is", "explain"],
}


def looks_like_math(segment: str) -> bool:
    """True if the segment contains a numeric arithmetic expression."""
    return bool(re.search(r"\d\s*[\+\-\*/%]\s*\d", segment))


def split_into_segments(query: str) -> list:
    """Break a compound query into segments on 'and', 'then', or commas."""
    parts = re.split(r"\band then\b|\bthen\b|\band\b|,", query, flags=re.IGNORECASE)
    return [p.strip() for p in parts if p.strip()]


def classify_segment(segment: str) -> str:
    """Return the tool name that best matches a query segment."""
    seg_lower = segment.lower()
    if any(k in seg_lower for k in INTENT_KEYWORDS["calculator"]) or looks_like_math(segment):
        return "calculator"
    if any(k in seg_lower for k in INTENT_KEYWORDS["keywords"]):
        return "keywords"
    if any(k in seg_lower for k in INTENT_KEYWORDS["summarize"]):
        return "summarize"
    if any(k in seg_lower for k in INTENT_KEYWORDS["knowledge"]):
        return "knowledge"
    return "general"


def plan(query: str) -> list:
    """Produce an ordered list of {step, tool, input} dicts for a query."""
    segments = split_into_segments(query)
    if not segments:
        segments = [query]

    steps = []
    for i, seg in enumerate(segments, start=1):
        tool = classify_segment(seg)
        steps.append({"step": i, "tool": tool, "input": seg})
    return steps


## ⚙️ Step 4: Tool executor

Runs each planned step against the right tool, with per-step try/except so one bad step doesn't abort the rest of the plan.

In [7]:
# ⚙️ TOOL EXECUTOR

def extract_expression(text: str) -> str:
    """Pull the numeric expression out of a calculator-intent segment."""
    match = re.search(r"[0-9][0-9\.\s\+\-\*\/\(\)%]*[0-9\)]", text)
    return match.group(0).strip() if match else text


LEADING_TRIGGERS = [
    "extract keywords from", "extract keywords", "keywords from", "keywords",
    "summarize", "summarise", "tl;dr", "shorten",
]


def extract_topic(text: str) -> str:
    """Pull the topic/text a segment refers to.

    Prefers content after 'from'/'of'/'about'; otherwise strips a known
    leading trigger phrase (e.g. 'Summarize ...') so it isn't treated as content.
    """
    lower = text.lower()
    for marker in ["from", "of", "about"]:
        if marker in lower:
            idx = lower.index(marker) + len(marker)
            return text[idx:].strip()

    for trigger in LEADING_TRIGGERS:
        if lower.startswith(trigger):
            return text[len(trigger):].strip()

    return text


def execute_step(step: dict) -> dict:
    """Run one planned step and return a result dict with status."""
    tool = step["tool"]
    raw_input = step["input"]

    try:
        if tool == "calculator":
            expression = extract_expression(raw_input)
            output = calculator(expression)
            status = "error" if output.startswith("Error") else "ok"

        elif tool == "keywords":
            output = extract_keywords(extract_topic(raw_input))
            status = "ok"

        elif tool == "summarize":
            output = summarize(extract_topic(raw_input))
            status = "ok"

        elif tool == "knowledge":
            # e.g. "define agent" / "explain llm" / "what is a python"
            words = [w for w in re.split(r"\W+", raw_input.lower()) if w]
            term = words[-1] if words else raw_input
            output = knowledge_lookup(term)
            status = "ok"

        else:  # general fallback
            output = f"No specific tool matched; noted your request: '{raw_input}'"
            status = "ok"

    except Exception as e:
        output = f"Unexpected error: {e}"
        status = "error"

    return {**step, "output": output, "status": status}


def execute_plan(steps: list) -> list:
    """Execute every step in a plan, collecting results in order."""
    return [execute_step(step) for step in steps]


## 📦 Step 5: Response formatter + Agent orchestrator

Ties planner, executor, and memory together, and produces one structured JSON object per turn:

```
{
  "type": "single" | "multi" | "error",
  "plan": [...],
  "steps": [...],
  "result": ...
}
```

In [8]:
# 📦 RESPONSE FORMATTER + AGENT

def format_response(steps: list) -> dict:
    """Combine executed steps into a single structured response."""
    if not steps:
        return {"type": "error", "plan": [], "steps": [], "result": "No steps produced."}

    any_error = any(s["status"] == "error" for s in steps)

    if len(steps) == 1:
        response_type = "error" if any_error else steps[0]["tool"]
        result = steps[0]["output"]
    else:
        response_type = "multi_error" if any_error else "multi"
        result = [{"tool": s["tool"], "output": s["output"]} for s in steps]

    return {
        "type": response_type,
        "plan": [{"step": s["step"], "tool": s["tool"], "input": s["input"]} for s in steps],
        "steps": steps,
        "result": result,
    }


class AgenticPipeline:
    """The full agent: plan -> execute -> log to memory -> format response."""

    def __init__(self, verbose: bool = True):
        self.memory = Memory()
        self.verbose = verbose

    def run(self, query: str) -> dict:
        steps_plan = plan(query)
        executed = execute_plan(steps_plan)
        response = format_response(executed)
        self.memory.log(query, steps_plan, executed, response)

        if self.verbose:
            for s in executed:
                print(f"[LOG] step={s['step']} tool={s['tool']} status={s['status']}")

        return response


## 🧪 Test cases

Includes single-tool queries **and** a compound, multi-step query.

In [9]:
# 🧪 TEST CASES

agent = AgenticPipeline(verbose=True)

test_queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Summarize The quarterly report showed strong revenue growth. Costs also rose sharply. Investors reacted positively.",
    "Define agent",
    "Calculate 12 * 4 and extract keywords from The stock market saw significant volatility today",
]

for q in test_queries:
    print("Query:", q)
    print("Response:", agent.run(q))
    print("-" * 60)


Query: Calculate 20 + 5
[LOG] step=1 tool=calculator status=ok
Response: {'type': 'calculator', 'plan': [{'step': 1, 'tool': 'calculator', 'input': 'Calculate 20 + 5'}], 'steps': [{'step': 1, 'tool': 'calculator', 'input': 'Calculate 20 + 5', 'output': '25', 'status': 'ok'}], 'result': '25'}
------------------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
[LOG] step=1 tool=keywords status=ok
Response: {'type': 'keywords', 'plan': [{'step': 1, 'tool': 'keywords', 'input': 'Extract keywords from Artificial Intelligence is transforming industries'}], 'steps': [{'step': 1, 'tool': 'keywords', 'input': 'Extract keywords from Artificial Intelligence is transforming industries', 'output': ['artificial', 'intelligence', 'transforming', 'industries'], 'status': 'ok'}], 'result': ['artificial', 'intelligence', 'transforming', 'industries']}
------------------------------------------------------------
Query: What is machi

## 🧠 Inspect memory

Every turn is logged, so you can audit the full trace of any past query.

In [10]:
# Inspect the last 2 turns stored in memory
for turn in agent.memory.recent(2):
    print(turn["timestamp"], "->", turn["query"])
    print("  plan:", turn["plan"])
    print()


2026-07-19T14:07:21 -> Define agent
  plan: [{'step': 1, 'tool': 'knowledge', 'input': 'Define agent'}]

2026-07-19T14:07:21 -> Calculate 12 * 4 and extract keywords from The stock market saw significant volatility today
  plan: [{'step': 1, 'tool': 'calculator', 'input': 'Calculate 12 * 4'}, {'step': 2, 'tool': 'keywords', 'input': 'extract keywords from The stock market saw significant volatility today'}]



## 🎯 Interactive mode

Run this cell to chat with the pipeline. Type `exit` to stop.

In [ ]:
# 🎯 Interactive mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent.run(user_input))


[LOG] step=1 tool=general status=ok
Response: {'type': 'general', 'plan': [{'step': 1, 'tool': 'general', 'input': ''}], 'steps': [{'step': 1, 'tool': 'general', 'input': '', 'output': "No specific tool matched; noted your request: ''", 'status': 'ok'}], 'result': "No specific tool matched; noted your request: ''"}
[LOG] step=1 tool=calculator status=ok
Response: {'type': 'calculator', 'plan': [{'step': 1, 'tool': 'calculator', 'input': '2+3'}], 'steps': [{'step': 1, 'tool': 'calculator', 'input': '2+3', 'output': '5', 'status': 'ok'}], 'result': '5'}
[LOG] step=1 tool=general status=ok
Response: {'type': 'general', 'plan': [{'step': 1, 'tool': 'general', 'input': 'hi how are you?'}], 'steps': [{'step': 1, 'tool': 'general', 'input': 'hi how are you?', 'output': "No specific tool matched; noted your request: 'hi how are you?'", 'status': 'ok'}], 'result': "No specific tool matched; noted your request: 'hi how are you?'"}
[LOG] step=1 tool=calculator status=ok
Response: {'type': 'calcul